In [150]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import concurrent.futures
import os
import random
import json
import threading
from concurrent.futures import ThreadPoolExecutor
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib3.exceptions import TimeoutError, ReadTimeoutError
from PIL import Image

In [151]:
workers = os.cpu_count()
workers

24

In [152]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [153]:
def setup_driver():
    """
    Initializes and configures the Selenium Chrome WebDriver.
    
    This function sets up Chrome options to automatically allow location access
    and uses webdriver-manager to handle the driver installation.
    
    Returns:
        webdriver.Chrome: The configured WebDriver instance, or None if initialization fails.
    """
    try:
        print("🚀 Initializing WebDriver...")
        options = webdriver.ChromeOptions()
        options.add_experimental_option("prefs", {
            "profile.default_content_setting_values.geolocation": 1  # 1: Allow, 2: Block
        })
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
        print("👍 WebDriver successfully initialized.")
        return driver
    except Exception as e:
        print(f"❌ Failed to initialize WebDriver: {e}")
        return None

In [154]:
def initial_navigation(driver, url):
    """
    Navigates to the GoFood website and reaches the 'Terdekat' (Nearest) category page.
    
    Args:
        driver (webdriver.Chrome): The active WebDriver instance.
        url (str): The GoFood URL to open.
    """
    print(f"\n🌍 Opening URL: {url}")
    driver.get(url)

    print("🔎 Finding and clicking the location input field...")
    location_input = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.ID, "location-picker"))
    )
    location_input.click()
    print("👍 Location input field clicked.")

    print("📍 Clicking 'Use your current location' button...")
    use_current_location_button = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.XPATH, '//*[contains(text(), "Pakai lokasimu saat ini")]'))
    )
    use_current_location_button.click()
    print("👍 'Use your current location' button clicked.")

    print("🧭 Clicking the 'Explore' button...")
    explore_button = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(., "Eksplor")]'))
    )
    explore_button.click()
    print("👍 'Explore' button clicked.")
    
    time.sleep(5) 

    print("🏠 Finding and clicking the 'Terdekat' category...")
    terdekat_category_button = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.XPATH, "//h3[@title='Terdekat']"))
    )
    terdekat_category_button.click()
    print("👍 'Terdekat' category successfully clicked.")

In [155]:
def scroll_and_load_all_data(driver):
    """
    Dynamically scrolls the page and clicks 'Load more' to ensure all restaurants are loaded.
    
    Args:
        driver (webdriver.Chrome): The active WebDriver instance.
    """
    print("\n🔄 Starting dynamic scroll to load all restaurants...")
    scroll_attempts = 0
    max_scroll_attempts_without_button = 5

    while scroll_attempts < max_scroll_attempts_without_button:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        print("   Scrolling down...")
        time.sleep(2)

        try:
            load_more_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, '//button[.//span[text()="Muat lebih banyak"]]'))
            )
            load_more_button.click()
            print("   👍 'Load more' button found and clicked.")
            scroll_attempts = 0
        except TimeoutException:
            scroll_attempts += 1
            print(f"   ⏳ 'Load more' button not found (Attempt {scroll_attempts}/{max_scroll_attempts_without_button}).")

    print("\n✅ Scrolling finished. Assuming all data is now loaded.")

In [156]:
def scrape_restaurant_list(driver):
    """
    Parses the page source and extracts the initial list of restaurants.
    
    Args:
        driver (webdriver.Chrome): The active WebDriver instance.
        
    Returns:
        list: A list of dictionaries, where each dictionary contains the initial details of one restaurant.
    """
    print("\n🍽️ Waiting for the restaurant list to be fully present...")
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'my-6')]"))
    )
    print("👍 Restaurant list is present.")
    
    print("📄 Getting and parsing page source code...")
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')

    scraped_data = []
    restaurant_container = soup.find('div', class_=lambda c: c and 'my-6' in c.split())

    if not restaurant_container:
        print("❌ Could not find the main restaurant container on the page.")
        return []
    
    restaurant_cards = restaurant_container.find_all('a', recursive=False)
    print(f"✅ Found {len(restaurant_cards)} restaurants. Processing initial data...")

    for card in restaurant_cards:
        name_tag = card.find('p', class_='gf-label-m')
        name = name_tag['title'].strip() if name_tag and name_tag.has_attr('title') else 'N/A'
        
        link_href = card.get('href', '')
        full_link = f"https://gofood.co.id{link_href}" if link_href and link_href.startswith('/') else link_href
        
        if name != 'N/A' and full_link:
            scraped_data.append({
                'Nama Restoran': name,
                'Link': full_link
            })
            
    return scraped_data

In [157]:
def scrape_reviews(driver, restaurant_name):
    """
    Scrapes all loaded reviews from the review page.
    """
    print("   - Scraping reviews...")
    all_reviews_data = []
    
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'flex flex-col space-y-10')]"))
        )
    except TimeoutException:
        print("   - ⚠️ No review container found on the page.")
        return []

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    review_container = soup.find('div', class_=lambda c: c and 'flex' in c and 'flex-col' in c and 'space-y-10' in c)
    
    if not review_container:
        print("   - ⚠️ Could not find review container.")
        return []

    review_cards = review_container.find_all('div', recursive=False)
    print(f"   - Found {len(review_cards)} reviews to parse.")

    for card in review_cards:
        try:
            name_tag = card.find('h3', class_='text-gf-content-secondary gf-label-m')
            name = name_tag.text.strip() if name_tag else 'N/A'

            since_tag = card.find('span', class_='mt-1 text-gf-content-muted gf-body-xs md:gf-body-s')
            since = since_tag.text.replace('Pengguna Gojek sejak', '').strip() if since_tag else 'N/A'

            rating_tag = card.find('span', class_='ml-1 inline-block')
            rating = rating_tag.text.strip() if rating_tag else 'N/A'
            
            review_text_tag = card.find('p', class_='break-words gf-body-m')
            review_text = review_text_tag.text.strip() if review_text_tag else ''

            product_tag = card.find('span', class_='ml-2 break-words md:mt-1')
            products = product_tag.text.strip() if product_tag else 'N/A'
            
            bought_date_tag = card.find('div', class_='mt-4 text-gf-content-muted gf-body-s')
            bought_date = bought_date_tag.text.replace('Dibeli tanggal', '').strip() if bought_date_tag else 'N/A'
            
            all_reviews_data.append({
                'Nama Restoran': restaurant_name,
                'Nama': name,
                'Pengguna Gojek Sejak': since,
                'Rating': rating,
                'Ulasan': review_text,
                'Produk yang Dibeli': products,
                'Tanggal Beli': bought_date,
                'Waktu Scraping': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
        except Exception as e:
            print(f"   - ❗️ Error parsing a review card: {e}")
            continue

    return all_reviews_data

In [158]:
def scrape_menu(driver, restaurant_name):
    print("   - 📜 Scraping menu...")
    all_menu_data = []
    
    try:
        page_soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        potential_cards = page_soup.select('div.mt-4.md\\:mx-2, div.items-stretch.justify-between, a[href*="/gofood/pesan"]')
        print(f"   - Found {len(potential_cards)} potential menu cards to analyze.")

        for card in potential_cards:
            name_tag = card.find('h3', class_='text-gf-content-primary')
            if not name_tag:
                continue 

            name = name_tag.get_text(strip=True)
            
            detail_tag = card.find('p', class_='text-gf-content-muted')
            detail = detail_tag.get_text(strip=True) if detail_tag else 'N/A'
            
            price = 'N/A'
            price_spans = card.find_all('span')
            for span in price_spans:
                potential_price = span.get_text(strip=True).replace('.', '').replace('Rp', '').strip()
                if potential_price.isdigit():
                    price = span.get_text(strip=True)
                    break 

            if name and price != 'N/A':
                all_menu_data.append({
                    'Nama Restoran': restaurant_name,
                    'Nama Menu': name,
                    'Detail Menu': detail,
                    'Harga': price,
                    'Waktu Scraping': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                })
                
    except Exception as e:
        print(f"   - ❗️ An error occurred while parsing the menu item: {e}")

    print(f"   - ✅ Success scrape {len(all_menu_data)} item menu.")
    return all_menu_data

In [159]:
def scrape_promos(driver, restaurant_name):
    print("   - 🎟️ Searching and extracting promotions...")
    all_promo_data = []

    try:
        lihat_semua_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//div[contains(., 'promo')]//button[.//span[text()='Lihat semua']]"))
        )
        lihat_semua_button.click()
        print("   - The “Lihat semua ” promo button is clicked.")

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'space-y-3.5')]"))
        )
        time.sleep(1) 

        page_soup = BeautifulSoup(driver.page_source, 'html.parser')
        promo_cards = page_soup.select('div.ineligible > div, div.eligible > div')

        for card in promo_cards:
            title_tag = card.find('div', class_='gf-label-l')
            usage_tag = card.find('div', class_='gf-label-xs')
            
            if title_tag and usage_tag:
                title = title_tag.get_text(strip=True)
                usage = usage_tag.get_text(strip=True)
                
                detail_items = card.find_all('li', class_='gf-body-s')
                details = ", ".join([item.get_text(strip=True) for item in detail_items]) if detail_items else 'N/A'

                all_promo_data.append({
                    'Nama Restoran': restaurant_name,
                    'Judul Promo': title,
                    'Cara Menggunakan': usage,
                    'Detail Promo': details,
                    'Waktu Scraping': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                })
        
        driver.find_element(By.TAG_NAME, 'body').send_keys(webdriver.common.keys.Keys.ESCAPE)
        print(f"   - ✅ Successfully scraped {len(all_promo_data)} promotions. Pop-up closed.")

    except TimeoutException:
        print("   - ℹ️ There is no ‘View all’ button for the promotions found.")
    except Exception as e:
        print(f"   - ❗️ An error occurred while parsing the promo: {e}")

    return all_promo_data

In [160]:
def get_full_restaurant_details_and_reviews(driver, initial_data_list):
    detailed_results = []
    all_reviews_results = []
    all_menu_results = []
    all_promos_results = []
    total_restaurants = len(initial_data_list)
    
    for i, restaurant in enumerate(initial_data_list):
        print(f"\n[{i+1}/{total_restaurants}] 🔎 Processing: {restaurant['Nama Restoran']}")
        
        try:
            driver.get(restaurant['Link'])
            print("   - Loading restaurant detail page...")
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, "//a[text()='Jarak']"))
            )
            time.sleep(2)

            page_soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            rating, distance, price_str, price_detail = 'N/A', 'N/A', 'N/A', 'N/A'
            address, opening_hours = "Alamat tidak ditemukan", {}
            
            info_panel = page_soup.find('div', class_=lambda c: c and 'mr-12' in c and 'inline-flex' in c)
            if info_panel:
                rating_container = info_panel.find('svg', class_=lambda c: c and 'text-gf-support-warning-default' in c)
                if rating_container:
                    rating_tag = rating_container.find_next_sibling('p')
                    rating = rating_tag.text.strip() if rating_tag else rating
                
                distance_container = info_panel.find('svg', class_=lambda c: c and 'text-gf-brand-retail-red' in c)
                if distance_container:
                    distance_tag = distance_container.find_next_sibling('p')
                    distance = distance_tag.text.strip() if distance_tag else distance
                
                price_level_tag = info_panel.find('div', attrs={'data-testid': 'priceLevel'})
                if price_level_tag:
                    active_dollars = price_level_tag.find_all('div', class_='text-gf-content-primary')
                    price_level = len(active_dollars)
                    price_str = f"{'$' * price_level}{'·' * (4 - price_level)}"
                    price_detail_container = price_level_tag.find_parent('div').find_next_sibling('div')
                    if price_detail_container:
                        price_detail_tag = price_detail_container.find('span')
                        price_detail = price_detail_tag.text.strip() if price_detail_tag else price_detail

            print(f"   - Main details: Rating: {rating}, Distance: {distance}, Price: {price_str}, Detail: {price_detail}")

            print("   - Clicking 'Jarak' button for address...")
            jarak_button = driver.find_element(By.XPATH, "//a[text()='Jarak']")
            jarak_button.click()
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//h2[starts-with(@id, 'headlessui-dialog-title-')]")))
            time.sleep(1)
            
            popup_soup = BeautifulSoup(driver.page_source, 'html.parser')
            address_tag = popup_soup.find('div', class_='text-gf-content-muted gf-body-s')
            address = address_tag.text.strip() if address_tag else address
            hours_container = popup_soup.find('h4', string='Jam buka')
            if hours_container:
                day_elements = hours_container.find_next_siblings('div')
                for day_element in day_elements:
                    day_tag = day_element.find('div', class_=lambda c: c and 'gf-label-s' in c)
                    hour_tag = day_element.find('div', class_='text-left')
                    if day_tag and hour_tag:
                        opening_hours[day_tag.text.strip()] = hour_tag.text.strip()
            
            driver.find_element(By.TAG_NAME, 'body').send_keys(webdriver.common.keys.Keys.ESCAPE)
            time.sleep(1)
            
            print(f"   ✅ Success: Scraped details for {restaurant['Nama Restoran']}")

            menu_items = scrape_menu(driver, restaurant['Nama Restoran'])
            if menu_items:
                all_menu_results.extend(menu_items) 

            promo_items = scrape_promos(driver, restaurant['Nama Restoran'])
            if promo_items:
                all_promos_results.extend(promo_items)
            
            detailed_results.append({
                'Nama Restoran': restaurant['Nama Restoran'], 'Rating': rating, 'Jarak': distance,
                'Tingkat Harga': price_str, 'Detail Harga': price_detail, 'Alamat': address,
                'Jam Buka (Senin)': opening_hours.get('Senin', 'Tutup'), 'Jam Buka (Selasa)': opening_hours.get('Selasa', 'Tutup'),
                'Jam Buka (Rabu)': opening_hours.get('Rabu', 'Tutup'), 'Jam Buka (Kamis)': opening_hours.get('Kamis', 'Tutup'),
                'Jam Buka (Jumat)': opening_hours.get('Jumat', 'Tutup'), 'Jam Buka (Sabtu)': opening_hours.get('Sabtu', 'Tutup'),
                'Jam Buka (Minggu)': opening_hours.get('Minggu', 'Tutup'), 'Link': restaurant['Link'],
                'Waktu Scraping': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })

            try:
                print("   - Finding and clicking 'Cek ulasan'...")
                cek_ulasan_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, "//a[text()='Cek ulasan']"))
                )
                cek_ulasan_button.click()
                print("   - Navigated to reviews page.")
                
                time.sleep(3)

                for click_count in range(5):
                    try:
                        print(f"   - Attempting to click 'Muat lebih banyak' ({click_count+1}/2)...")
                        load_more_reviews_button = WebDriverWait(driver, 5).until(
                            EC.element_to_be_clickable((By.XPATH, '//button[.//span[text()="Muat lebih banyak"]]'))
                        )
                        driver.execute_script("arguments[0].click();", load_more_reviews_button)
                        print("   - 'Muat lebih banyak' clicked.")
                        time.sleep(3) 
                    except (TimeoutException, ElementClickInterceptedException):
                        print("   - 'Muat lebih banyak' button not found or not clickable. Assuming all reviews are loaded.")
                        break 
                
                reviews = scrape_reviews(driver, restaurant['Nama Restoran'])
                if reviews:
                    all_reviews_results.extend(reviews)
                    print(f"   - ✅ Successfully scraped {len(reviews)} reviews.")

            except (TimeoutException, NoSuchElementException):
                print(f"   - ⚠️ Could not find or click 'Cek ulasan' for {restaurant['Nama Restoran']}.")
            except Exception as e:
                print(f"   - ❌ An error occurred during review scraping: {e}")

        except Exception as e:
            print(f"   ❌ An unexpected error occurred for {restaurant['Nama Restoran']}: {e}")

            detailed_results.append({
                'Nama Restoran': restaurant['Nama Restoran'], 'Rating': 'GAGAL', 'Jarak': 'GAGAL', 'Tingkat Harga': 'GAGAL',
                'Detail Harga': 'GAGAL', 'Alamat': 'GAGAL', 'Jam Buka (Senin)': 'GAGAL', 'Jam Buka (Selasa)': 'GAGAL',
                'Jam Buka (Rabu)': 'GAGAL', 'Jam Buka (Kamis)': 'GAGAL', 'Jam Buka (Jumat)': 'GAGAL',
                'Jam Buka (Sabtu)': 'GAGAL', 'Jam Buka (Minggu)': 'GAGAL', 'Link': restaurant['Link']
            })
        
        time.sleep(1)

    df_restaurants = pd.DataFrame(detailed_results)
    df_reviews = pd.DataFrame(all_reviews_results)
    df_menu = pd.DataFrame(all_menu_results)
    df_promos = pd.DataFrame(all_promos_results)

    # 3. Kembalikan empat DataFrame tersebut
    return df_restaurants, df_reviews, df_menu, df_promos

In [161]:
def display_results(data, title):
    """
    Converts scraped data into a Pandas DataFrame and prints it with a title.
    """
    if not data.empty:
        print(f"\n✅ --- {title} ---")
        try:
            from IPython.display import display
            display(data)
        except (ImportError, NameError):
            print(data.to_string())
    else:
        print(f"\n❌ No data was successfully scraped for '{title}'.")


In [162]:
driver = None
try:
    driver = setup_driver()
    if driver:
        gofood_url = "https://gofood.co.id/id"
        initial_navigation(driver, gofood_url)
        scroll_and_load_all_data(driver)
        
        initial_restaurant_list = scrape_restaurant_list(driver)
        
        if initial_restaurant_list:
            detailed_data, reviews_data, menu_data, promo_data = get_full_restaurant_details_and_reviews(driver, initial_restaurant_list)
            
            # Display both dataframes
            display_results(detailed_data, "Restaurant Details Scraping Results")
            detailed_data.to_csv('hasil_detail_restoran.csv', index=False)
            display_results(reviews_data, "Customer Reviews Scraping Results")
            reviews_data.to_csv('hasil_ulasan_pelanggan.csv', index=False)
            display_results(menu_data, "Restaurant Menu Scraping Results")
            menu_data.to_csv('hasil_menu_restoran.csv', index=False)
            display_results(promo_data, "Restaurant Promotions Scraping Results")
            promo_data.to_csv('hasil_promo_restoran.csv', index=False)

            # df_detailed = pd.DataFrame(detailed_data)
            # display_results(df_detailed, "Restaurant Details Scraping Results")
            # df_detailed.to_csv('hasil_detail_restoran.csv', index=False)
            
            # # Konversi dan simpan ulasan pelanggan
            # df_reviews = pd.DataFrame(reviews_data)
            # display_results(df_reviews, "Customer Reviews Scraping Results")
            # df_reviews.to_csv('hasil_ulasan_pelanggan.csv', index=False)
            
            # # Konversi dan simpan menu restoran
            # df_menu = pd.DataFrame(menu_data)
            # display_results(df_menu, "Restaurant Menu Scraping Results")
            # df_menu.to_csv('hasil_menu_restoran.csv', index=False)
            
            # # Konversi dan simpan promo restoran
            # df_promo = pd.DataFrame(promo_data)
            # display_results(df_promo, "Restaurant Promotions Scraping Results")
            # df_promo.to_csv('hasil_promo_restoran.csv', index=False)
        else:
            print("No restaurants were found to process further.")

except Exception as e:
    print(f"\n❌ An unexpected error occurred during the main process: {e}")
    
finally:
    if driver:
        print("\n🚪 Process finished, closing the browser.")
        driver.quit()

🚀 Initializing WebDriver...
👍 WebDriver successfully initialized.

🌍 Opening URL: https://gofood.co.id/id
🔎 Finding and clicking the location input field...
👍 Location input field clicked.
📍 Clicking 'Use your current location' button...
👍 'Use your current location' button clicked.
🧭 Clicking the 'Explore' button...
👍 'Explore' button clicked.
🏠 Finding and clicking the 'Terdekat' category...
👍 'Terdekat' category successfully clicked.

🔄 Starting dynamic scroll to load all restaurants...
   Scrolling down...
   ⏳ 'Load more' button not found (Attempt 1/5).
   Scrolling down...
   ⏳ 'Load more' button not found (Attempt 2/5).
   Scrolling down...
   ⏳ 'Load more' button not found (Attempt 3/5).
   Scrolling down...
   ⏳ 'Load more' button not found (Attempt 4/5).
   Scrolling down...
   ⏳ 'Load more' button not found (Attempt 5/5).

✅ Scrolling finished. Assuming all data is now loaded.

🍽️ Waiting for the restaurant list to be fully present...
👍 Restaurant list is present.
📄 Getting 

,Nama Restoran,Rating,Jarak,Tingkat Harga,Detail Harga,Alamat,Jam Buka (Senin),Jam Buka (Selasa),Jam Buka (Rabu),Jam Buka (Kamis),Jam Buka (Jumat),Jam Buka (Sabtu),Jam Buka (Minggu),Link,Waktu Scraping
0,"DIMSUM MBLEDOS, Merr",4.8,0.48 km,$$··,16rb-40rb,"Jl. Dr Ir H Soekarno No. 53, Mulyorejo, Surabaya",06:00-23:00,06:00-23:00,06:00-23:00,06:00-23:00,06:00-23:00,06:00-23:00,06:00-23:00,https://gofood.co.id/surabaya/restaurant/dimsu...,2025-07-04 21:22:35
1,Angsle & Ronde 66,5,0.49 km,$$··,16rb-40rb,"Jl. Ploso Baru No. 189, Tambaksari, Surabaya",15:00-22:00,15:00-22:00,15:00-22:00,15:00-22:00,15:00-22:00,15:00-22:00,Tutup,https://gofood.co.id/surabaya/restaurant/angsl...,2025-07-04 21:23:10
2,"De Bronx (Mieyabi Hot), Ploso",5,0.77 km,$$··,16rb-40rb,"Jl. Ploso Baru No. 16, Tambaksari, Surabaya",15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,https://gofood.co.id/surabaya/restaurant/de-br...,2025-07-04 21:23:43
3,"Terang Bulan Bangka C.N.K, Karang Empat Besar",4.9,1 km,$$$·,40rb-100rb,"Jl. Karang Empat Besar 102, Tambaksari, Surabaya",07:00-22:30,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,08:00-22:00,10:00-22:00,https://gofood.co.id/surabaya/restaurant/teran...,2025-07-04 21:24:16
4,"Surabaya Patata, Dharmahusada",4.9,1.3 km,$$$·,40rb-100rb,"Jl. Dharmahusada No. 187, Gubeng, Surabaya",07:00-21:45,07:00-21:45,07:00-21:30,07:00-21:45,07:00-21:45,07:00-21:45,07:00-21:45,https://gofood.co.id/surabaya/restaurant/surab...,2025-07-04 21:24:49
5,"Kebab Kings, Kenjeran",3.8,1.45 km,$$··,16rb-40rb,Jalan Kenjeran No 325 Surabaya (Teras Alfamidi...,13:30-21:30,13:30-21:30,Tutup,13:30-21:30,13:30-21:30,13:30-21:30,13:30-21:30,https://gofood.co.id/surabaya/restaurant/kebab...,2025-07-04 21:25:18
6,"Tahu Tek Pak Jayen, Dharmahusada",4.9,1.47 km,$$··,16rb-40rb,"Jl. Dharmahusada No. 112, Gubeng, Surabaya",16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,https://gofood.co.id/surabaya/restaurant/tahu-...,2025-07-04 21:25:41
7,"Malika Kebab 14, Tambaksari",4.5,1.49 km,$$··,16rb-40rb,"Jl. Bronggalan No. 38, Pacar Kembang, Tambaksa...",00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,https://gofood.co.id/surabaya/restaurant/malik...,2025-07-04 21:26:16
8,"Yung Ho, Dharmahusada",4.9,1.59 km,$$$·,40rb-100rb,"Jl. Dharmahusada No. 50, Gubeng, Surabaya Timur",00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,https://gofood.co.id/surabaya/restaurant/yung-...,2025-07-04 21:26:46
9,"Pizza Hut Delivery - PHD, Dharmahusada",4.9,1.69 km,$$$·,40rb-100rb,"JL. Dharmahusada, No. 115 E, Gubeng, Surabaya",05:30-23:15,05:30-23:15,05:30-23:15,05:30-23:15,07:30-23:15,07:30-23:15,07:30-23:15,https://gofood.co.id/surabaya/restaurant/pizza...,2025-07-04 21:27:15



✅ --- Customer Reviews Scraping Results ---


,Nama Restoran,Nama,Pengguna Gojek Sejak,Rating,Ulasan,Produk yang Dibeli,Tanggal Beli,Waktu Scraping
0,"DIMSUM MBLEDOS, Merr",Faris Alkaula,28 Oktober 2019,4.0,sumpit sama spicy cheese dan mayo nya ketingga...,Mie Lamian Kaldu Ayam - Dimsum Siomay Beef,1 Juli 2025,2025-07-04 21:22:55
1,"DIMSUM MBLEDOS, Merr",Rafa,18 September 2020,1.0,chili oilny gk ada. kecewa jd gk enk dimsumny ...,Dimsum Mentai/Hampers,23 Juni 2025,2025-07-04 21:22:55
2,"DIMSUM MBLEDOS, Merr",afifatun nikmah,21 Oktober 2018,5.0,"driver baik ,sabar",BUY 1 GET 1 All Dimsum (Jam 22:00-23:00),17 Mei 2025,2025-07-04 21:22:55
3,"DIMSUM MBLEDOS, Merr",Nowo Aprilianto,10 Mei 2016,1.0,Saya pesan 5 item. Dikirimnya cuma 4 item.,Diskon 50 Persen All Dimsum (Jam 06:00-07:00),16 Mei 2025,2025-07-04 21:22:55
4,"DIMSUM MBLEDOS, Merr",T****,23 April 2019,3.0,saus cheese nya kurang bgt.....,Dimsum Surf Roll - Dimsum Siomay Ayam Udang - ...,1 Mei 2025,2025-07-04 21:22:55
...,...,...,...,...,...,...,...,...
4768,"Ayam Bakar Primarasa, Kusuma Bangsa",Ervina,17 Agustus 2015,5.0,"enaaak bangeet, sering sering promo yaa",Ayam Bakar / Goreng (1 Ekor),23 Januari 2022,2025-07-04 21:53:18
4769,"Ayam Bakar Primarasa, Kusuma Bangsa",shirley,30 November 2015,5.0,cuminya enak tdk keras,Ayam Bakar / Goreng (paha / Dada) - Cumi Goren...,10 Januari 2022,2025-07-04 21:53:18
4770,"Ayam Bakar Primarasa, Kusuma Bangsa",S******,30 November 2015,5.0,enak enak enak enak enak,Ayam Bakar / Goreng (1 Ekor),12 Januari 2022,2025-07-04 21:53:18
4771,"Ayam Bakar Primarasa, Kusuma Bangsa",S******,30 November 2015,5.0,ayam gorengnya enak empuk,Cumi Goreng Tepung - Ayam Bakar / Goreng (paha...,12 Januari 2022,2025-07-04 21:53:18



✅ --- Restaurant Menu Scraping Results ---


,Nama Restoran,Nama Menu,Detail Menu,Harga,Waktu Scraping
0,"DIMSUM MBLEDOS, Merr",Dimsum Mentai/Hampers,Isi 20 pcs dimsum dengan topping saos mentai t...,162.900,2025-07-04 21:22:32
1,"DIMSUM MBLEDOS, Merr",Dimsum Siomay Salmon,"Ayam, udang, salmon (isi 4)",28.900,2025-07-04 21:22:32
2,"DIMSUM MBLEDOS, Merr",Dimsum Siomay Ayam Udang,"Ayam, udang (isi 4)",28.900,2025-07-04 21:22:32
3,"DIMSUM MBLEDOS, Merr",Dimsum Hakau,"Udang, kulit lembut (isi 4)",28.900,2025-07-04 21:22:32
4,"DIMSUM MBLEDOS, Merr",Dimsum Mantao,Roti khas china + susu kental manis (isi 3),28.900,2025-07-04 21:22:32
...,...,...,...,...,...
4882,"Ayam Bakar Primarasa, Kusuma Bangsa",Banana Cake,Banana Cake With Almond Slice And Chocolate Chips,68.000,2025-07-04 21:52:56
4883,"Ayam Bakar Primarasa, Kusuma Bangsa",Pisang Bolen,Pisang Bolen Hidangan Ringan Berbahan Baku Pis...,68.000,2025-07-04 21:52:56
4884,"Ayam Bakar Primarasa, Kusuma Bangsa",Mahabbah Hampers,6 pcs pia rasa kuno dengan kartu ucapan lebara...,84.286,2025-07-04 21:52:56
4885,"Ayam Bakar Primarasa, Kusuma Bangsa",Safa Hampers,"6 pcs pia rasa kuno, 1 toples kue sagu vanilla...",127.143,2025-07-04 21:52:56



✅ --- Restaurant Promotions Scraping Results ---


,Nama Restoran,Judul Promo,Cara Menggunakan,Detail Promo,Waktu Scraping
0,"DIMSUM MBLEDOS, Merr","Diskon makanan 35%, maks. 41rb",Use with GoPay atau 7 lainnya,Min. pembelian 65rb,2025-07-04 21:22:35
1,"DIMSUM MBLEDOS, Merr","Diskon makanan 35%, maks. 41rb",Use with GoPay atau 7 lainnya,Min. pembelian 78rb,2025-07-04 21:22:35
2,"DIMSUM MBLEDOS, Merr","Diskon makanan 30%, maks. 19rb",Use with GoPay atau 7 lainnya,Min. pembelian 35rb,2025-07-04 21:22:35
3,"DIMSUM MBLEDOS, Merr","Diskon makanan 30%, maks. 19rb",Use with GoPay atau 7 lainnya,Min. pembelian 42rb,2025-07-04 21:22:35
4,"Terang Bulan Bangka C.N.K, Karang Empat Besar","Diskon makanan 45%, maks. 157rb",Use with GoPay atau 6 lainnya,Min. pembelian 144rb,2025-07-04 21:24:16
...,...,...,...,...,...
107,"Martabak Gading Pecenongan, Jl. Kertajaya","Diskon makanan 30%, maks. 62rb",Use with GoPay atau 7 lainnya,Min. pembelian 115rb,2025-07-04 21:52:28
108,"Martabak Gading Pecenongan, Jl. Kertajaya","Diskon makanan 25%, maks. 27rb",Use with GoPay atau 7 lainnya,Min. pembelian 60rb,2025-07-04 21:52:28
109,"Ayam Bakar Primarasa, Kusuma Bangsa","Diskon makanan 30%, maks. 151rb",Use with GoPay atau 7 lainnya,Min. pembelian 280rb,2025-07-04 21:52:58
110,"Ayam Bakar Primarasa, Kusuma Bangsa","Diskon makanan 25%, maks. 63rb",Use with GoPay atau 7 lainnya,Min. pembelian 140rb,2025-07-04 21:52:58



🚪 Process finished, closing the browser.
